# geodetic-engine -- Catalog examples

`geodetic_engine.catalog` lists what the active PROJ database holds -- CRSs, coordinate operations, datums and datum ensembles, ellipsoids, prime meridians, units, operation methods, coordinate systems and grid files -- and filters them with one consistent set of keyword arguments.

Each listing function returns a lazy `Listing`: nothing is read until it is used, and the entries can be looped over, turned into a pandas DataFrame, or exported as PROJJSON-shaped JSON. Every entry's `to_json_dict()` is the object's official PROJJSON, as PROJ exports it.

The catalog lists what the registry *publishes*. To see what PROJ would *run* between two CRSs, use `geodetic_engine.geodesy.available_operations`; Section 4 compares the two.

## 1. Which database answers

Every listing reads the `proj.db` PROJ resolves codes against. `database()` names it, with the dataset versions it holds, and `authorities()` lists the authorities any `authority=` filter may name.

In [ ]:
from geodetic_engine import catalog

info = catalog.database()
print("PROJ", info.proj_version, "| EPSG", info.epsg_version)
print([str(authority) for authority in catalog.authorities()])

## 2. Projected CRSs on a datum, for an area

The question the catalog answers most often. `area_of_interest` is `(west, south, east, north)` in degrees, and an area of use intersects it exactly when PROJ's own CRS search says so. This is the example from the issue that asked for the catalog:

In [ ]:
european = catalog.crs(
    authority="EPSG",
    kind="projected",
    datum="ETRS89",
    area_of_interest=(-10, 50, 30, 72),
)
print(len(european), "projected ETRS89 CRSs intersect the area")
european.to_dataframe()[["authority_code", "name", "projection_method"]].head(10)

Narrowed to UTM zones whose area of use *contains* a box around Bergen, with a pattern on the name:

In [ ]:
import re

bergen = (5.0, 60.2, 5.6, 60.6)
for entry in catalog.crs(
    authority="EPSG",
    kind="projected",
    datum="ETRS89",
    name=re.compile(r"UTM zone \d+N$"),
    area_of_interest=bergen,
    contains=True,
):
    print(entry, "|", entry.area_of_use.name[:60])

### Engineering CRSs

Local CRSs, such as plant, site and seismic grids, have the kind `"engineering"`. pyproj's own CRS search reports them as a plain `CRS`, and its engineering filter finds none; the catalog reads each CRS's kind from `proj.db`, so this finds them all:

In [ ]:
from collections import Counter

engineering = catalog.crs(kind="engineering")
print(len(engineering), "engineering CRSs:", Counter(e.auth_name for e in engineering))
engineering.to_dataframe()[["authority_code", "name", "datums", "area_of_use"]]

### Every CRS of one authority

`authority=` takes any authority `catalog.authorities()` lists. The CRSs the OSDU catalogue defines itself are under the authority `OSDU` in the database the released package ships. They are bound CRSs, each carrying the transformation to WGS 84 that the catalogue binds it to, and each lists the OSDU record ids that publish it:

In [ ]:
osdu = catalog.crs(authority="OSDU")
print(len(osdu), "CRSs of authority OSDU:", Counter(e.kind.value for e in osdu))
print(sum(e.bound for e in osdu), "of them bound")
osdu.to_dataframe()[["authority_code", "name", "kind", "bound", "osdu_ids"]].head(10)

The catalogue also publishes plain EPSG CRSs, under record ids of their own, and those stay under the authority `EPSG`. Every entry carries the record ids that publish it in `osdu_ids`, so every CRS the OSDU catalogue publishes, whatever its authority, is:

In [ ]:
published = [entry for entry in catalog.crs(deprecated=None) if entry.osdu_ids]
print(len(published), "have OSDU record ids:", Counter(e.auth_name for e in published))
ed50_utm31 = catalog.crs(code="EPSG:23031").one()
print(ed50_utm31, ed50_utm31.osdu_ids)

## 3. Transformations between two CRSs

`operations()` with `source_crs` and `target_crs` lists the operations registered between them, each with its method, stated accuracy, version label and area of use. The second example from the issue:

In [ ]:
ed50 = catalog.operations(
    source_crs="EPSG:4230", target_crs="EPSG:4326", max_accuracy=5.0
)
print(len(ed50), "ED50 -> WGS 84 transformations state 5 m or better")
ed50.to_dataframe()[
    ["authority_code", "method", "accuracy", "version", "area_of_use"]
].head(12)

In [ ]:
by_grid = catalog.operations(
    source_crs="EPSG:4230", target_crs="EPSG:4326", grid_based=True
)
for entry in by_grid:
    print(
        f"{entry.authority_code:10} {entry.method.name:12} grids={entry.grid_names}  available={entry.grids_available()}"
    )

Choosing one for a point offshore Norway: the most accurate transformation whose area of use contains it.

In [ ]:
point = (4.1279, 63.5850)
candidates = catalog.operations(
    source_crs="EPSG:4230",
    target_crs="EPSG:4326",
    area_of_interest=(*point, *point),
    contains=True,
    max_accuracy=10,
)
best = min(candidates, key=lambda entry: entry.accuracy)
print("chosen:", best, best.accuracy, "m")
print("valid in:", best.area_of_use.name)

That is a UK offshore transformation: its bounding box reaches the point, but its area of use, in words, does not. A bounding box is the rectangle around an area, so read the area's name before trusting it. Adding `area_name` settles it, and the entry's code is an `operation=` for `transform()`:

In [ ]:
from geodetic_engine.geodesy import transform

best = min(
    catalog.operations(
        source_crs="EPSG:4230",
        target_crs="EPSG:4326",
        area_of_interest=(*point, *point),
        contains=True,
        area_name="Norway",
    ),
    key=lambda entry: entry.accuracy,
)
print("chosen:", best, best.accuracy, "m |", best.area_of_use.name)
transform("EPSG:4230", "EPSG:4326", point, operation=best.authority_code).coordinates

## 4. Registered, or assembled by PROJ

EPSG registers datum transformations between geographic CRSs. Between two projected CRSs nothing is registered, but PROJ assembles a chain: the inverse projection, a registered transformation, the projection. The catalog answers the first question, `available_operations` the second:

In [ ]:
from geodetic_engine.geodesy import available_operations

print(
    "registered:",
    len(catalog.operations(source_crs="EPSG:23031", target_crs="EPSG:32631")),
)
for candidate in available_operations("EPSG:23031", "EPSG:32631")[:3]:
    print("assembled: ", candidate.references, candidate.accuracy, "m")

PROJ never offers a deprecated operation. The catalog lists them on request, with what replaces them:

In [ ]:
for entry in catalog.operations(
    source_crs="EPSG:4230", target_crs="EPSG:4326", deprecated=True
):
    print(entry, "-> replaced by", entry.replaced_by or "nothing stated")

## 5. Datums, ensembles and frames

`datums()` lists geodetic, vertical and engineering datums and datum ensembles. An ensemble lists its members, and `member_of` turns that round:

In [ ]:
etrs89 = catalog.datums(code="EPSG:6258").one()
print(
    etrs89.projjson_type,
    "| accuracy",
    etrs89.ensemble_accuracy,
    "m |",
    len(etrs89.members),
    "members",
)
[str(member) for member in catalog.datums(member_of="ETRS89")][:8]

In [ ]:
dynamic = catalog.datums(
    authority="EPSG", dynamic=True, frame_reference_epoch=(2014, None)
)
[(str(d), d.frame_reference_epoch) for d in dynamic]

## 6. Following references

Every entry points at the objects it is built from with an `ObjectRef`, and every `code` filter accepts one, so following a reference is one more listing. From a CRS to its datum, to its ellipsoid, to every other datum on that ellipsoid:

In [ ]:
crs = catalog.crs(code="EPSG:23031").one()
datum = catalog.datums(code=crs.datums).one()
ellipsoid = catalog.ellipsoids(code=datum.ellipsoid).one()
others = catalog.datums(ellipsoid=ellipsoid)
print(crs, "->", datum, "->", ellipsoid)
print(
    f"a={ellipsoid.semi_major_axis} m, 1/f={ellipsoid.inverse_flattening}; {len(others)} datums use it"
)

## 7. Ellipsoids, prime meridians and units

In [ ]:
catalog.ellipsoids(authority="EPSG", semi_major_axis=(6378100, 6378200)).to_dataframe()[
    ["authority_code", "name", "semi_major_axis", "inverse_flattening"]
]

In [ ]:
catalog.prime_meridians(authority="EPSG", longitude=(-20, 20)).to_dataframe()[
    ["name", "longitude", "unit", "longitude_degrees"]
]

In [ ]:
catalog.units(authority="EPSG", kind="linear", name="foot").to_dataframe()[
    ["authority_code", "name", "conversion_factor"]
]

## 8. Methods and coordinate systems

A method lists the parameters PROJ states for it, and how many operations apply it. A coordinate system lists its axes; `axes` finds the northing-first ones, whose CRSs are easiest to get wrong:

In [ ]:
tm = catalog.methods(code="EPSG:9807").one()
print(tm, "|", tm.operation_count, "operations")
print([p.name for p in tm.parameters])
[
    str(cs)
    for cs in catalog.coordinate_systems(
        authority="EPSG", axes=("N", "E"), unit="metre"
    )
]

## 9. Grid files

`grids()` lists the grid files operations read, under the file PROJ reads for each, and whether it is installed. Here, the grids for Norway, and the operations that need a missing one:

In [ ]:
norway = catalog.grids(area_name="Norway")
norway.to_dataframe()[["name", "kind", "installed", "used_by"]]

In [ ]:
missing = [grid for grid in norway if not grid.installed]
codes = [code for grid in missing for code in grid.used_by]
if codes:
    print([str(entry) for entry in catalog.operations(code=codes, deprecated=None)])
else:
    print("every grid these operations read is installed")

`check_remote=True` also asks PROJ's content delivery network which grids it publishes. It is the only listing that uses the network, so it is not run here.

## 10. Exporting

A listing exports as JSON: PROJJSON-shaped items in a small envelope recording the database that answered and the filters asked for. `include_definitions=True` adds each entry's full PROJJSON. The envelope's JSON Schema ships with the package as `catalog.listing_schema()`.

In [ ]:
import json

document = catalog.operations(code="EPSG:1612").to_json_dict()
print(json.dumps(document["items"][0], indent=2))
print(document["filters"], document["database"]["metadata"]["EPSG.VERSION"])

In [ ]:
definition = catalog.operations(code="EPSG:1612").one().to_json_dict()
print(definition["$schema"])
[(parameter["name"], parameter["value"]) for parameter in definition["parameters"]]